# Potato Disease Classification - MobileNetV2 Training

Train a new MobileNetV2 model for potato disease classification.

Based on: https://github.com/saiimmani/crop-disease-prediction-model

Classes: Early Blight, Late Blight, Healthy

Runtime: GPU recommended (Runtime -> Change runtime type -> T4 GPU)

## Step 0: Setup and Dependencies

In [ ]:
import os
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import numpy as np
import json

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Step 1: Download PlantVillage Dataset

In [ ]:
# Download PlantVillage dataset
# Option 1: Clone from GitHub
!git clone https://github.com/spMohanty/PlantVillage-Dataset.git /tmp/plantvillage_repo 2>/dev/null

# Check structure
!ls /tmp/plantvillage_repo/ 2>/dev/null | head -10
!find /tmp/plantvillage_repo -type d -name '*otato*' 2>/dev/null | head -5

In [ ]:
# Prepare potato-only dataset directory
import shutil

# Create PlantVillage directory structure for potato classes
DATA_DIR = Path('/content/PlantVillage')
DATA_DIR.mkdir(exist_ok=True)

# Find and copy potato classes
potato_classes = ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']

# Search for potato directories
found = False
for root, dirs, files in os.walk('/tmp/plantvillage_repo'):
    for d in dirs:
        if d in potato_classes:
            src = os.path.join(root, d)
            dst = DATA_DIR / d
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"Copied: {d} ({len(list(Path(dst).iterdir()))} images)")
            found = True

if not found:
    print("ERROR: Could not find potato classes!")
    print("Trying alternative download...")
    # Alternative: download a smaller version
!wget -q "https://raw.githubusercontent.com/spMohanty/PlantVillage-Dataset/master/raw/color/Potato___Early_blight" -O /tmp/test 2>/dev/null || echo "Alternative download not available"

print(f"\nDataset ready at: {DATA_DIR}")
!ls {DATA_DIR}

## Step 2: Load Dataset (Same as train_transfer_learning.py)

In [ ]:
# Constants (same as train_transfer_learning.py)
IMAGE_SIZE = 256
BATCH_SIZE = 32
CHANNELS = 3
INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
N_CLASSES = 3
EPOCHS_HEAD = 6
EPOCHS_FINE_TUNE = 6

# Load dataset using same method as train_transfer_learning.py
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    str(DATA_DIR),
    shuffle=True,
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

class_names = dataset.class_names
print(f"Classes: {class_names}")
print(f"Total batches: {len(dataset)}")

In [ ]:
# Split dataset into train/val/test (same as train_transfer_learning.py)
def get_dataset_partitions_tf(ds, train_split=0.8, val_split=0.1, test_split=0.1, shuffle=True, shuffle_size=10000):
    assert (train_split + test_split + val_split) == 1
    ds_size = len(ds)
    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)
    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)
    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)
    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = get_dataset_partitions_tf(dataset)

print(f"Train batches: {len(train_ds)}")
print(f"Val batches: {len(val_ds)}")
print(f"Test batches: {len(test_ds)}")

In [ ]:
# Preprocessing function for MobileNetV2
def preprocess_fn(image, label):
    return preprocess_input(image), label

train_ds = train_ds.map(preprocess_fn).cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess_fn).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess_fn).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

print("Dataset preprocessed!")

In [ ]:
# Visualize sample images
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        # Denormalize for display
        img = (images[i].numpy() + 1) / 2 * 255
        plt.imshow(img.astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: Build MobileNetV2 Model

In [ ]:
# Data augmentation (same as train_transfer_learning.py)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
])

# Build MobileNetV2 model
base_model = MobileNetV2(
    input_shape=INPUT_SHAPE,
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=INPUT_SHAPE)
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(N_CLASSES, activation='softmax')(x)
model = models.Model(inputs=inputs, outputs=outputs)

model.summary()

## Step 4: Phase 1 - Train Classification Head

In [ ]:
# Compile model (same as train_transfer_learning.py)
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Callbacks
phase1_callbacks = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(
        filepath='/tmp/mobilenetv2_phase1_best.weights.h5',
        save_best_only=True, save_weights_only=True, monitor='val_accuracy'
    )
]

print("Phase 1: training head with frozen base model...")

history1 = model.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    verbose=1,
    epochs=EPOCHS_HEAD,
    callbacks=phase1_callbacks
)

## Step 5: Phase 2 - Fine-tune Base Model

In [ ]:
# Unfreeze top layers of base model (same as train_transfer_learning.py)
base_model.trainable = True

# Freeze first 100 layers, train the rest
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print(f"Phase 2: fine-tuning last {sum(1 for l in base_model.layers if l.trainable)} layers...")

phase2_callbacks = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(
        filepath='/tmp/mobilenetv2_phase2_best.weights.h5',
        save_best_only=True, save_weights_only=True, monitor='val_accuracy'
    )
]

history2 = model.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    verbose=1,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=phase2_callbacks
)

## Step 6: Evaluate Model

In [ ]:
# Evaluate on test set (same as train_transfer_learning.py)
scores = model.evaluate(test_ds)
print(f"\nTest loss: {scores[0]:.4f}")
print(f"Test accuracy: {scores[1]:.4f}")

In [ ]:
# Plot training history
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc, label='Training Accuracy')
axes[0].plot(val_acc, label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(loss, label='Training Loss')
axes[1].plot(val_loss, label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions
plt.figure(figsize=(15, 10))
for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        img = (images[i].numpy() + 1) / 2 * 255
        plt.imshow(img.astype("uint8"))
        true_label = class_names[labels[i]]
        pred_label = class_names[pred_classes[i]]
        confidence = np.max(preds[i]) * 100
        color = 'green' if true_label == pred_label else 'red'
        plt.title(f"True: {true_label}\nPred: {pred_label}\n({confidence:.1f}%)", color=color)
        plt.axis("off")

plt.suptitle("Model Predictions", fontsize=14)
plt.tight_layout()
plt.show()

## Step 7: Grad-CAM Visualization

In [ ]:
import matplotlib.cm as cm

def compute_gradcam(model, image, class_index, layer_name=None):
    if layer_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                layer_name = layer.name
                break
    
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    
    img_tensor = tf.expand_dims(image, 0)
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        loss = predictions[:, class_index]
    
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    
    return heatmap.numpy()

def create_gradcam_overlay(image, heatmap, alpha=0.4):
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis],
        (image.shape[0], image.shape[1])
    ).numpy().squeeze()
    
    heatmap_gamma = np.power(np.clip(heatmap_resized, 0, 1), 0.7)
    colored = cm.jet(heatmap_gamma)[:, :, :3]
    colored_255 = (colored * 255).astype(np.uint8)
    
    # Denormalize image for display
    img_display = ((image + 1) / 2 * 255).astype(np.float32)
    overlay = (img_display * (1 - alpha) + colored_255.astype(np.float32) * alpha)
    
    return np.clip(overlay, 0, 255).astype(np.uint8)

# Visualize Grad-CAM
plt.figure(figsize=(16, 12))
for images, labels in test_ds.take(1):
    for i in range(min(6, len(images))):
        image = images[i]
        true_label = class_names[labels[i]]
        
        pred = model.predict(tf.expand_dims(image, 0), verbose=0)
        pred_class = np.argmax(pred[0])
        pred_conf = pred[0][pred_class] * 100
        pred_label = class_names[pred_class]
        
        heatmap = compute_gradcam(model, image, pred_class)
        overlay = create_gradcam_overlay(image, heatmap)
        
        ax = plt.subplot(4, 3, i * 3 + 1)
        img_display = ((image.numpy() + 1) / 2 * 255).astype("uint8")
        plt.imshow(img_display)
        plt.title(f"Original\nTrue: {true_label}")
        plt.axis('off')
        
        ax = plt.subplot(4, 3, i * 3 + 2)
        plt.imshow(overlay)
        plt.title(f"Grad-CAM\nPred: {pred_label} ({pred_conf:.1f}%)")
        plt.axis('off')
        
        ax = plt.subplot(4, 3, i * 3 + 3)
        plt.imshow(heatmap, cmap='jet')
        plt.title("Heatmap")
        plt.axis('off')

plt.suptitle("Grad-CAM Visualization", fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Save Model

In [ ]:
# Save model (same format as train_transfer_learning.py)
model.save(str(Path('/tmp/saved_models/mobilenetv2')))
print("MobileNetV2 model saved to /tmp/saved_models/mobilenetv2")

# Also save as H5 for compatibility
model.save(str(Path('/tmp/saved_models/mobilenetv2.h5')))
print("MobileNetV2 model saved to /tmp/saved_models/mobilenetv2.h5")

# Save training results
with open(str(Path('/tmp/saved_models/_mobilenetv2_results.txt')), 'w') as f:
    f.write(f"Test loss: {scores[0]:.4f}\n")
    f.write(f"Test accuracy: {scores[1]:.4f}\n")
    best_val_acc = max(history2.history['val_accuracy'])
    f.write(f"Best validation accuracy (phase 2): {best_val_acc:.4f}\n")
    f.write(f"Phase 1 epochs: {len(history1.history['loss'])}\n")
    f.write(f"Phase 2 epochs: {len(history2.history['loss'])}\n")
print("Training results saved to /tmp/saved_models/_mobilenetv2_results.txt")

In [ ]:
# Download model from Colab
from google.colab import files
print("Downloading model files...")
files.download('/tmp/saved_models/mobilenetv2.h5')
print("\nDownload complete!")
print("\nTo use in your project:")
print("1. Replace saved_models/3/model.h5 with the downloaded mobilenetv2.h5")
print("2. Restart the backend server")

## Summary

### Training Results
- Architecture: MobileNetV2 + Custom Head
- Parameters: ~2.5M
- Two-phase training: Head then fine-tune (same as train_transfer_learning.py)
- Saved to: /tmp/saved_models/

### Next Steps
1. Download mobilenetv2.h5 from above
2. Replace saved_models/3/model.h5
3. Restart backend server
4. Test with Grad-CAM notebook